# 0.0 Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import Lasso
from sklearn import metrics as mt

# 0.1 - Load Datasets

In [2]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [3]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 2 — Treino com parâmetros default

In [ ]:
# Instanciar o modelo com parâmetros default
model_def = Lasso()

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 3 — Performance no treino (default)

In [ ]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

## Passo 4 — Performance na validação (default)

In [ ]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

## Passo 5 — Ajuste de hiperparâmetros

In [4]:
print("--- Testando Múltiplos Hiperparâmetros na Regressão Lasso (Corrigido) ---")
melhor_alpha = 1.0
melhor_max_iter = 1000
melhor_r2_val = -999

list_alpha = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
list_max_iter = [1000, 2000, 5000]

for alpha in list_alpha:
    for mi in list_max_iter:
        
        # Instancia o Lasso com os parâmetros do loop
        model = Lasso(alpha=alpha, max_iter=mi, random_state=42)
        
        # O try/except agora serve para pular APENAS problemas matemáticos reais de convergência brutos,
        # mas as variáveis e métricas foram limpas.
        try:
            model.fit(X_train, y_train.values.ravel())
            
            yhat_train = model.predict(X_train)
            yhat_val = model.predict(X_val)
            
            r2_train = mt.r2_score(y_train, yhat_train)
            r2_val = mt.r2_score(y_val, yhat_val)
            rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
            mae_val = mt.mean_absolute_error(y_val, yhat_val)
            
            # Blindagem do MAPE: calcula em formato bruto primeiro
            mape_raw = mt.mean_absolute_percentage_error(y_val, yhat_val)
            # Se o MAPE for infinito ou absurdamente grande devido aos zeros, tratamos como string
            mape_print = f"{mape_raw * 100:.2f}%" if np.isfinite(mape_raw) and mape_raw < 1000 else "Distort (High)"
            
            print(f"Alpha: {alpha:<6} | Max_Iter: {mi:4d} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f} | MAPE Val: {mape_print}")
            
            if r2_val > melhor_r2_val:
                melhor_r2_val = r2_val
                melhor_alpha = alpha
                melhor_max_iter = mi
                
        except Exception as e:
            # Se falhar, agora ele te avisa o MOTIVO real do erro no print
            print(f"Erro real na combinação Alpha {alpha} | Max_Iter {mi}: {e}")

print("=" * 80)
print(f"-> VENCEDOR DO ENSAIO LASSO:")
print(f"Melhor Alpha: {melhor_alpha}")
print(f"Melhor Max_Iter: {melhor_max_iter}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")
    

--- Testando Múltiplos Hiperparâmetros na Regressão Lasso (Corrigido) ---
Alpha: 0.0001 | Max_Iter: 1000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.26%
Alpha: 0.0001 | Max_Iter: 2000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.26%
Alpha: 0.0001 | Max_Iter: 5000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.26%
Alpha: 0.001  | Max_Iter: 1000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.29%
Alpha: 0.001  | Max_Iter: 2000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.29%
Alpha: 0.001  | Max_Iter: 5000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.29%
Alpha: 0.01   | Max_Iter: 1000 | R² Treino: 0.0459 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.62%
Alpha: 0.01   | Max_Iter: 2000 | R² Treino: 0.0459 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.62%
Alpha: 0.01   | Max_Iter: 5000 | R² Treino: 0.0459 | R² Val: 0.0399 | RMSE Val

## Passo 6 — União treino + validação

In [ ]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

## Passo 7 — Retreino com melhores parâmetros

In [ ]:
alpha = 0.001
max_iter = 1000

# Treinando o modelo para ver as outras métricas de avaliação
model = Lasso(alpha=alpha, max_iter=max_iter, random_state=42)
model.fit(X_train, y_train.values.ravel())

# Previsão de treino e validação
yhat_train = model.predict(X_train)
yhat_val = model.predict(X_val)

#Performance Train
r2_train = mt.r2_score(y_train, yhat_train)
mse_train = mt.mean_squared_error(y_train, yhat_train)
rmse_train = np.sqrt(mse_train)
mae_train = mt.mean_absolute_error(y_train, yhat_train)
mape_train = mt.mean_absolute_percentage_error(y_train, yhat_train)

#Performance Validation
r2_val = mt.r2_score(y_val, yhat_val)
mse_val = mt.mean_squared_error(y_val, yhat_val)
rmse_val = np.sqrt(mse_val)
mae_val = mt.mean_absolute_error(y_val, yhat_val)
mape_val = mt.mean_absolute_percentage_error(y_val, yhat_val)

# Métrica de Treinamento e Teste
print(f"MÉTRICAS TREINAMENTO:")
print(f"R² (Coef. Determinação): {r2_train:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_train:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_train:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_train:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_train * 100:.2f}%")
print("=" * 65)
print(f"MÉTRICAS VALDAÇÃO:")
print(f"R² (Coef. Determinação): {r2_val:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_val:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_val:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_val:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_val * 100:.2f}%")


MÉTRICAS TREINAMENTO:
R² (Coef. Determinação): 0.0461
MSE (Erro Quadrático Médio): 456.00
RMSE (Raiz do Erro Quadrático): 21.35
MAE (Erro Absoluto Médio): 17.00
MAPE (Erro Percentual Médio): 865.39%
MÉTRICAS VALDAÇÃO:
R² (Coef. Determinação): 0.0399
MSE (Erro Quadrático Médio): 458.44
RMSE (Raiz do Erro Quadrático): 21.41
MAE (Erro Absoluto Médio): 17.04
MAPE (Erro Percentual Médio): 868.29%


## Passo 8 — Performance no teste

In [6]:
# ==============================================================================
# JUNTAR AS BASES E EXECUTAR O TESTE FINAL
# ==============================================================================
print("--- Executando o Teste Final ---")

# Juntando treino e validação para dar mais volume de dados ao KNN
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

# Treinando o modelo definitivo com o melhor K encontrado
model_final = Lasso(alpha=melhor_alpha, max_iter=melhor_max_iter, random_state=42)
model_final.fit(X_train_final, y_train_final.values.ravel())

# Previsão final usando o dataset de teste (que ficou isolado o tempo todo)
yhat_test = model_final.predict(X_test)

#Performance Test
r2_test = mt.r2_score(y_test, yhat_test)
mse_test = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

# Métrica final de produção
print(f"MÉTRICAS NO DATASET DE TESTE:")
print(f"R² (Coef. Determinação): {r2_test:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_test:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_test:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_test:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_test * 100:.2f}%")
print("=" * 65)

--- Executando o Teste Final ---
MÉTRICAS NO DATASET DE TESTE:
R² (Coef. Determinação): 0.0511
MSE (Erro Quadrático Médio): 462.00
RMSE (Raiz do Erro Quadrático): 21.49
MAE (Erro Absoluto Médio): 17.14
MAPE (Erro Percentual Médio): 853.31%


## Passo 9 — Avaliar e registrar 3 insights

**Insight 1 — [Título do Insight]**
> [Descreva aqui o insight mais relevante observado no comportamento deste algoritmo com estes dados.]

**Insight 2 — [Título do Insight]**
> [Descreva aqui o segundo insight: diferença treino vs validação, comportamento dos hiperparâmetros, etc.]

**Insight 3 — [Título do Insight]**
> [Descreva aqui o terceiro insight: comparação com outras abordagens, limitações, ou pontos de atenção.]

## Passo 10 — Quadro Comparativo — Diagnóstico Completo

In [ ]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   '-', '-', r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, '-', '-', rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  '-', '-', mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%', '-', '-', f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))